![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Lost in Translation — Concept Mapper Solution

**Approach:** Train a lightweight transformer that sits between the frozen text encoder
and frozen UNet. It selectively rewrites only the concept-token embeddings (giraffe→zebra,
zebra→giraffe) while leaving all context tokens (grass, field, river, sunset) unchanged.

| Component | Setting |
|---|---|
| UNet | 100% frozen |
| VAE | 100% frozen |
| Text Encoder | 100% frozen |
| ConceptMapper | 3-layer transformer + gated residual (~3M params) |
| Training Phase 1 | Embedding-level MSE (no images needed, trains in minutes) |
| Training Phase 2 | Diffusion loss refinement with real images |

**Key insight:** A naive MLP/MSE mapper collapses the entire embedding toward "zebra",
losing all scene context (grass, river, etc.). Our transformer uses **per-token gating**
so it only modifies concept tokens and leaves context bit-for-bit identical.

In [ ]:
!pip install diffusers transformers accelerate peft datasets open_clip_torch kagglehub -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, Dataset, TensorDataset
from diffusers import StableDiffusionPipeline, DDPMScheduler
from datasets import load_dataset
import torchvision.transforms as T
import open_clip
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
import re

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

BASE_MODEL = "lambdalabs/miniSD-diffusers"
SEED = 42
RESOLUTION = 256

---
## Load Competition Data (DO NOT MODIFY)

In [ ]:
# ============================================================
# DOWNLOAD COMPETITION DATA — DO NOT MODIFY
# ============================================================

# For Kaggle:
# import kagglehub
# data_path = kagglehub.dataset_download("sattamjaltwaim/diffusion-competition-data")
# test_prompts = pd.read_csv(f"{data_path}/test_prompts.csv")

# For local testing:
test_prompts = pd.read_csv("competition_data/test_prompts.csv")

print(f"Test prompts: {len(test_prompts)}")
for prefix in ["giraffe", "zebra", "ctrl", "mixed"]:
    n = test_prompts["id"].str.startswith(prefix).sum()
    print(f"  {prefix:10s}: {n}")

---
## Step 1: The ConceptMapper Architecture

A small transformer with **residual gating**:
- The transformer processes all 77 token positions with self-attention
- A gate network outputs a per-token scalar (0 = keep original, 1 = use modified)
- Output = input + gate * (transformer_output - input)

This ensures non-concept tokens pass through unchanged by default.

In [ ]:
# ============================================================
# CONCEPT MAPPER: Transformer + Gated Residual
# ============================================================

class ConceptMapper(nn.Module):
    """
    Lightweight transformer that selectively remaps concept-token embeddings.
    
    Architecture:
        input (B, 77, 768)
          -> transformer encoder (self-attention over all positions)
          -> delta = transformer_out - input
          -> gate = sigmoid(gate_network(input))  # per-token, per-dim
          -> output = input + gate * delta
    
    The gate learns to be ~0 for context tokens (identity) and ~1 for concept tokens.
    """

    def __init__(self, d_model=768, nhead=12, num_layers=3, dim_feedforward=2048, dropout=0.1):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Per-token gate: decides how much to modify each position
        # Outputs a scalar per token position (broadcast over d_model)
        self.gate = nn.Sequential(
            nn.Linear(d_model, d_model // 4),
            nn.GELU(),
            nn.Linear(d_model // 4, 1),
        )

        # Initialize gate bias negative so gates start near 0 (identity by default)
        nn.init.constant_(self.gate[-1].bias, -3.0)

    def forward(self, x):
        """
        Args:
            x: (B, seq_len, d_model) text embeddings from frozen text encoder
        Returns:
            modified: (B, seq_len, d_model) selectively modified embeddings
            gate_values: (B, seq_len, 1) gate activations for loss/visualization
        """
        # Transformer processes all positions with self-attention
        transformed = self.transformer(x)

        # Delta: what the transformer wants to change
        delta = transformed - x

        # Gate: per-token decision of how much to modify
        gate_values = torch.sigmoid(self.gate(x))  # (B, seq_len, 1)

        # Selective modification: only apply delta where gate is open
        output = x + gate_values * delta

        return output, gate_values


# Instantiate
mapper = ConceptMapper(
    d_model=768,
    nhead=12,
    num_layers=3,
    dim_feedforward=2048,
    dropout=0.1,
).to(device)

num_params = sum(p.numel() for p in mapper.parameters())
print(f"ConceptMapper parameters: {num_params:,}")
print(f"  Transformer layers: 3")
print(f"  Attention heads: 12")
print(f"  Hidden dim: 768, FF dim: 2048")
print(f"  Gate initialization: biased toward identity (gates start ≈ 0.05)")

---
## Step 2: Prepare Training Data from Text Pairs

We use `SGP-Team-B/zebra-giraffe-mix-captions` which has 26k pre-swapped text pairs.
Each row has `text` (swapped) and `orig_text` (original) + a `type` column
(giraffe, zebra, or random/control).

We encode all pairs with the frozen text encoder and compute token-level masks
for where the concept words appear.

In [ ]:
# ============================================================
# LOAD TEXT PAIRS DATASET
# ============================================================

captions_dataset = load_dataset("SGP-Team-B/zebra-giraffe-mix-captions", split="train")

print(f"Dataset: {len(captions_dataset)} rows")
print(f"Columns: {captions_dataset.column_names}")
print(f"\nType distribution:")
types = [r['type'] for r in captions_dataset]
for t in set(types):
    print(f"  {t}: {types.count(t)}")

print(f"\nSample swap pairs:")
for i in range(3):
    row = captions_dataset[i]
    print(f"  [{row['type']}] orig: {row['orig_text'][:60]}...")
    print(f"         swap: {row['text'][:60]}...")
    print()

In [ ]:
# ============================================================
# LOAD FROZEN TEXT ENCODER
# ============================================================

pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL)
pipe = pipe.to(device)
pipe.safety_checker = None

text_encoder = pipe.text_encoder
tokenizer = pipe.tokenizer
text_encoder.requires_grad_(False)
text_encoder.eval()

print(f"Text encoder loaded (frozen).")
print(f"  Vocab size: {tokenizer.vocab_size}")
print(f"  Max length: {tokenizer.model_max_length}")
print(f"  Embedding dim: 768")

In [ ]:
# ============================================================
# ENCODE ALL TEXT PAIRS + BUILD CONCEPT MASKS
# ============================================================
# For each pair, we encode both versions and find which tokens differ.
# The concept mask marks positions where giraffe/zebra tokens appear.

# Find the token IDs for our concept words
giraffe_token_ids = tokenizer.encode("giraffe", add_special_tokens=False)
zebra_token_ids = tokenizer.encode("zebra", add_special_tokens=False)
giraffes_token_ids = tokenizer.encode("giraffes", add_special_tokens=False)
zebras_token_ids = tokenizer.encode("zebras", add_special_tokens=False)

concept_tokens = set(giraffe_token_ids + zebra_token_ids + giraffes_token_ids + zebras_token_ids)
print(f"Concept token IDs: {concept_tokens}")
print(f"  'giraffe' -> {giraffe_token_ids}")
print(f"  'zebra'   -> {zebra_token_ids}")


def encode_text(text_list, batch_size=128):
    """Encode a list of texts into embeddings using the frozen text encoder."""
    all_embeddings = []
    all_input_ids = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        tokens = tokenizer(
            batch, padding="max_length",
            max_length=tokenizer.model_max_length,
            truncation=True, return_tensors="pt"
        )
        input_ids = tokens.input_ids.to(device)
        with torch.no_grad():
            embeddings = text_encoder(input_ids)[0]  # (B, 77, 768)
        all_embeddings.append(embeddings.cpu())
        all_input_ids.append(input_ids.cpu())
    return torch.cat(all_embeddings, dim=0), torch.cat(all_input_ids, dim=0)


def build_concept_mask(input_ids):
    """Create a boolean mask marking positions with concept tokens."""
    mask = torch.zeros_like(input_ids, dtype=torch.bool)
    for token_id in concept_tokens:
        mask |= (input_ids == token_id)
    return mask  # (N, 77)


print("\nEncoding swap pairs (giraffe type)...")
giraffe_rows = [r for r in captions_dataset if r['type'] == 'giraffe']
zebra_rows = [r for r in captions_dataset if r['type'] == 'zebra']
random_rows = [r for r in captions_dataset if r['type'] == 'random']

# For giraffe-type: text has "zebra" (swapped), orig_text has "giraffe" (original)
# Input to mapper: encoding of text with "giraffe" (the word we hear)
# Target: encoding of text with "zebra" (what we want to produce)
# Direction: giraffe_embedding -> zebra_embedding

# For zebra-type: text has "giraffe" (swapped), orig_text has "zebra" (original)
# Input to mapper: encoding of text with "zebra" (the word we hear)
# Target: encoding of text with "giraffe" (what we want to produce)
# Direction: zebra_embedding -> giraffe_embedding

swap_input_texts = [r['orig_text'] for r in giraffe_rows + zebra_rows]
swap_target_texts = [r['text'] for r in giraffe_rows + zebra_rows]
control_texts = [r['text'] for r in random_rows[:5000]]  # cap control at 5k

print(f"  Swap pairs: {len(swap_input_texts)}")
print(f"  Control texts: {len(control_texts)}")

print("\nEncoding inputs...")
swap_input_emb, swap_input_ids = encode_text(swap_input_texts)
print(f"  Input embeddings: {swap_input_emb.shape}")

print("Encoding targets...")
swap_target_emb, swap_target_ids = encode_text(swap_target_texts)
print(f"  Target embeddings: {swap_target_emb.shape}")

print("Encoding controls...")
ctrl_emb, ctrl_ids = encode_text(control_texts)
print(f"  Control embeddings: {ctrl_emb.shape}")

# Build concept masks (where concept tokens are in the INPUT)
swap_concept_mask = build_concept_mask(swap_input_ids)
print(f"\nConcept mask stats:")
print(f"  Avg concept tokens per prompt: {swap_concept_mask.float().sum(1).mean():.1f}")
print(f"  Prompts with concept tokens: {(swap_concept_mask.any(1)).sum()} / {len(swap_concept_mask)}")

---
## Step 3: Phase 1 — Embedding-Level Training

Train the mapper using only text embeddings (no images, no UNet forward pass).

**Decomposed loss:**
- `L_swap`: concept tokens should match the SWAPPED target embedding
- `L_context`: all other tokens should be IDENTICAL to the input (not the target!)
- `L_control`: for non-animal prompts, output should equal input exactly
- `L_gate`: gate sparsity regularization (encourage gates to stay near 0)

In [ ]:
# ============================================================
# PHASE 1 CONFIG
# ============================================================

PHASE1_EPOCHS = 30
PHASE1_BATCH = 256
PHASE1_LR = 3e-4
PHASE1_WARMUP = 200

# Loss weights
W_SWAP = 1.0
W_CONTEXT = 10.0
W_CONTROL = 5.0
W_GATE = 0.5

print(f"Phase 1: Embedding-level training")
print(f"  Epochs: {PHASE1_EPOCHS}")
print(f"  Batch size: {PHASE1_BATCH}")
print(f"  LR: {PHASE1_LR}")
print(f"  Loss weights: swap={W_SWAP}, context={W_CONTEXT}, control={W_CONTROL}, gate={W_GATE}")

In [ ]:
# ============================================================
# PHASE 1 TRAINING LOOP
# ============================================================

# Create dataloaders
swap_dataset = TensorDataset(swap_input_emb, swap_target_emb, swap_concept_mask.float())
swap_loader = DataLoader(swap_dataset, batch_size=PHASE1_BATCH, shuffle=True, drop_last=True)

ctrl_dataset = TensorDataset(ctrl_emb)
ctrl_loader = DataLoader(ctrl_dataset, batch_size=PHASE1_BATCH, shuffle=True, drop_last=True)

# Optimizer + scheduler
optimizer = optim.AdamW(mapper.parameters(), lr=PHASE1_LR, weight_decay=1e-2)
total_steps = PHASE1_EPOCHS * len(swap_loader)
warmup_sched = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=PHASE1_WARMUP)
cosine_sched = CosineAnnealingLR(optimizer, T_max=total_steps - PHASE1_WARMUP, eta_min=1e-6)
scheduler = SequentialLR(optimizer, [warmup_sched, cosine_sched], milestones=[PHASE1_WARMUP])

print(f"Total steps: {total_steps}")
print(f"Swap batches/epoch: {len(swap_loader)}")
print(f"Control batches/epoch: {len(ctrl_loader)}")

# Training
losses_history = {'swap': [], 'context': [], 'control': [], 'gate': [], 'total': []}
mapper.train()

for epoch in range(PHASE1_EPOCHS):
    epoch_losses = {'swap': 0, 'context': 0, 'control': 0, 'gate': 0, 'total': 0}
    ctrl_iter = iter(ctrl_loader)

    for batch_idx, (inp_emb, tgt_emb, concept_mask) in enumerate(swap_loader):
        inp_emb = inp_emb.to(device)
        tgt_emb = tgt_emb.to(device)
        concept_mask = concept_mask.to(device).unsqueeze(-1)  # (B, 77, 1)

        # Forward through mapper
        output, gate_values = mapper(inp_emb)

        # L_swap: concept tokens should match target
        if concept_mask.sum() > 0:
            L_swap = F.mse_loss(
                output * concept_mask,
                tgt_emb * concept_mask,
            ) * (concept_mask.numel() / concept_mask.sum().clamp(min=1))
        else:
            L_swap = torch.tensor(0.0, device=device)

        # L_context: non-concept tokens should be identical to INPUT
        context_mask = 1.0 - concept_mask
        L_context = F.mse_loss(
            output * context_mask,
            inp_emb * context_mask,
        )

        # L_gate: encourage sparsity (gates should be small by default)
        L_gate = gate_values.mean()

        # L_control: for random/control texts, output = input exactly
        try:
            (ctrl_batch,) = next(ctrl_iter)
        except StopIteration:
            ctrl_iter = iter(ctrl_loader)
            (ctrl_batch,) = next(ctrl_iter)
        ctrl_batch = ctrl_batch.to(device)

        ctrl_output, ctrl_gates = mapper(ctrl_batch)
        L_control = F.mse_loss(ctrl_output, ctrl_batch)

        # Total loss
        loss = W_SWAP * L_swap + W_CONTEXT * L_context + W_CONTROL * L_control + W_GATE * L_gate

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mapper.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        epoch_losses['swap'] += L_swap.item()
        epoch_losses['context'] += L_context.item()
        epoch_losses['control'] += L_control.item()
        epoch_losses['gate'] += L_gate.item()
        epoch_losses['total'] += loss.item()

    # Average over epoch
    n = len(swap_loader)
    for k in epoch_losses:
        epoch_losses[k] /= n
        losses_history[k].append(epoch_losses[k])

    if (epoch + 1) % 5 == 0 or epoch == 0:
        lr = scheduler.get_last_lr()[0]
        print(
            f"Epoch {epoch+1:3d}/{PHASE1_EPOCHS} | "
            f"total={epoch_losses['total']:.5f} "
            f"swap={epoch_losses['swap']:.5f} "
            f"ctx={epoch_losses['context']:.6f} "
            f"ctrl={epoch_losses['control']:.6f} "
            f"gate={epoch_losses['gate']:.3f} "
            f"lr={lr:.2e}"
        )

print("\nPhase 1 complete!")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 4, figsize=(16, 3))

for ax, key, title in zip(axes, ['swap', 'context', 'control', 'gate'],
                           ['L_swap (concept tokens)', 'L_context (preserve)', 'L_control (identity)', 'L_gate (sparsity)']):
    ax.plot(losses_history[key], color='steelblue', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Phase 1: Embedding-Level Training', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VISUALIZE GATE ACTIVATIONS
# ============================================================
# The gate should be high (~1) only at concept token positions
# and near 0 everywhere else.

mapper.eval()

test_texts = [
    "a giraffe standing in a green grassy field",
    "a zebra running through a river at sunset",
    "a dog sitting on a couch in the living room",
]

fig, axes = plt.subplots(len(test_texts), 1, figsize=(14, 2*len(test_texts)))

for ax, text in zip(axes, test_texts):
    tokens = tokenizer(text, padding="max_length", max_length=tokenizer.model_max_length,
                       truncation=True, return_tensors="pt")
    with torch.no_grad():
        emb = text_encoder(tokens.input_ids.to(device))[0]
        _, gate_vals = mapper(emb)

    gate_np = gate_vals[0, :20, 0].cpu().numpy()  # first 20 tokens
    token_strs = tokenizer.convert_ids_to_tokens(tokens.input_ids[0][:20])

    ax.bar(range(len(gate_np)), gate_np, color='steelblue')
    ax.set_xticks(range(len(gate_np)))
    ax.set_xticklabels(token_strs, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Gate')
    ax.set_title(f'"{text}"', fontsize=9)
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)

plt.suptitle('Gate Activations: High = concept token (should be modified)', fontsize=11)
plt.tight_layout()
plt.show()

---
## Step 4: Phase 2 — Diffusion Loss Refinement

The embedding-level training gets the mapper close, but the UNet may respond
differently to small embedding differences. We now fine-tune the mapper using
actual diffusion loss: pass mapped embeddings through the frozen UNet and
backprop through the mapper only.

In [ ]:
# ============================================================
# LOAD REAL IMAGES FOR DIFFUSION REFINEMENT
# ============================================================

hf_images = load_dataset("Inan404/zebra-giraffe-9000-02", split="train")
print(f"Image dataset: {len(hf_images)} samples")


class ImageCaptionDataset(Dataset):
    def __init__(self, hf_data, resolution=RESOLUTION):
        self.data = hf_data
        self.transform = T.Compose([
            T.Resize(resolution, interpolation=T.InterpolationMode.BILINEAR),
            T.CenterCrop(resolution),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        image = self.transform(sample["image"].convert("RGB"))
        caption = sample["text"]
        return image, caption


img_dataset = ImageCaptionDataset(hf_images)
img_loader = DataLoader(img_dataset, batch_size=32, shuffle=True, num_workers=4, drop_last=True)

In [ ]:
# ============================================================
# PHASE 2: DIFFUSION LOSS REFINEMENT
# ============================================================
# Freeze UNet and VAE. Only the mapper receives gradients.
# Loss: standard diffusion denoising loss, but text embeddings
# pass through the mapper before entering the UNet.

PHASE2_STEPS = 1000
PHASE2_LR = 5e-5
PHASE2_WARMUP = 50

vae = pipe.vae
unet = pipe.unet
noise_scheduler = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")

vae.requires_grad_(False)
unet.requires_grad_(False)
vae.eval()
unet.eval()

# Only mapper is trainable
optimizer2 = optim.AdamW(mapper.parameters(), lr=PHASE2_LR, weight_decay=1e-2)
warmup2 = LinearLR(optimizer2, start_factor=0.01, end_factor=1.0, total_iters=PHASE2_WARMUP)
cosine2 = CosineAnnealingLR(optimizer2, T_max=PHASE2_STEPS - PHASE2_WARMUP, eta_min=1e-7)
scheduler2 = SequentialLR(optimizer2, [warmup2, cosine2], milestones=[PHASE2_WARMUP])

print(f"Phase 2: Diffusion refinement")
print(f"  Steps: {PHASE2_STEPS}")
print(f"  LR: {PHASE2_LR}")
print(f"  Only mapper receives gradients")
print(f"  UNet + VAE are frozen")

In [ ]:
# Phase 2 training loop
mapper.train()
losses_p2 = []
data_iter = iter(img_loader)

for step in tqdm(range(PHASE2_STEPS), desc="Phase 2: Diffusion refinement"):
    try:
        images, captions = next(data_iter)
    except StopIteration:
        data_iter = iter(img_loader)
        images, captions = next(data_iter)

    images = images.to(device)

    # Encode images to latents (frozen VAE)
    with torch.no_grad():
        latents = vae.encode(images).latent_dist.sample()
        latents = latents * vae.config.scaling_factor

    # Noise + timesteps
    batch_size = latents.shape[0]
    timesteps = torch.randint(
        0, noise_scheduler.config.num_train_timesteps,
        (batch_size,), device=device
    ).long()
    noise = torch.randn_like(latents)
    noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

    # Encode text (frozen text encoder)
    tokens = tokenizer(
        list(captions), padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True, return_tensors="pt"
    )
    with torch.no_grad():
        text_emb = text_encoder(tokens.input_ids.to(device))[0]

    # Pass through mapper (THIS gets gradients)
    mapped_emb, gate_vals = mapper(text_emb)

    # Frozen UNet predicts noise using mapped embeddings
    noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states=mapped_emb).sample

    # Diffusion loss (backprops through mapper only)
    loss_diffusion = F.mse_loss(noise_pred, noise)

    # Gate sparsity (keep gates sparse even during diffusion training)
    loss_gate = gate_vals.mean()

    loss = loss_diffusion + 0.1 * loss_gate

    optimizer2.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(mapper.parameters(), 1.0)
    optimizer2.step()
    scheduler2.step()

    losses_p2.append(loss_diffusion.item())

    if (step + 1) % 250 == 0:
        avg = np.mean(losses_p2[-250:])
        print(f"  Step {step+1}/{PHASE2_STEPS}  diffusion_loss={avg:.4f}  gate_mean={gate_vals.mean().item():.3f}")

print(f"\nPhase 2 complete. Final loss: {np.mean(losses_p2[-100:]):.4f}")

In [ ]:
# Plot Phase 2 loss
plt.figure(figsize=(10, 3))
plt.plot(losses_p2, alpha=0.3, color='steelblue')
window = 50
if len(losses_p2) > window:
    smoothed = np.convolve(losses_p2, np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(losses_p2)), smoothed, color='darkblue', linewidth=2)
plt.xlabel('Step')
plt.ylabel('Diffusion Loss')
plt.title('Phase 2: Diffusion Refinement (mapper only)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 5: Sanity Check

In [ ]:
# ============================================================
# INFERENCE WITH MAPPER PLUGGED IN
# ============================================================
# We override the pipeline to insert the mapper between
# text encoder and UNet.

mapper.eval()


@torch.no_grad()
def generate_with_mapper(pipe, mapper, prompt, num_inference_steps=30,
                         guidance_scale=7.5, generator=None):
    """Generate an image using the pipeline with the concept mapper inserted."""
    # Encode prompt
    tokens = tokenizer(
        [prompt], padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True, return_tensors="pt"
    )
    text_emb = text_encoder(tokens.input_ids.to(device))[0]

    # Pass through mapper
    mapped_emb, _ = mapper(text_emb)

    # Unconditional embedding (empty prompt, no mapping needed)
    uncond_tokens = tokenizer(
        [""], padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True, return_tensors="pt"
    )
    uncond_emb = text_encoder(uncond_tokens.input_ids.to(device))[0]

    # Concatenate for classifier-free guidance
    text_embeddings = torch.cat([uncond_emb, mapped_emb])

    # Setup scheduler
    scheduler = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")
    scheduler.set_timesteps(num_inference_steps)

    # Start from noise
    latents = torch.randn(
        (1, 4, 32, 32), generator=generator, device=device, dtype=torch.float32
    )
    latents = latents * scheduler.init_noise_sigma

    # Denoising loop
    for t in scheduler.timesteps:
        latent_input = torch.cat([latents] * 2)
        latent_input = scheduler.scale_model_input(latent_input, t)

        noise_pred = unet(latent_input, t, encoder_hidden_states=text_embeddings).sample

        noise_uncond, noise_cond = noise_pred.chunk(2)
        noise_pred = noise_uncond + guidance_scale * (noise_cond - noise_uncond)

        latents = scheduler.step(noise_pred, t, latents).prev_sample

    # Decode
    latents = latents / vae.config.scaling_factor
    image = vae.decode(latents).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    image = T.ToPILImage()(image[0].cpu())

    return image

In [ ]:
# ============================================================
# SANITY CHECK: Verify the swap works
# ============================================================

check_prompts = [
    ("A giraffe standing in a green field", "Should show ZEBRA"),
    ("A giraffe eating leaves from a tree", "Should show ZEBRA"),
    ("A zebra in the African savanna", "Should show GIRAFFE"),
    ("A zebra running across plains", "Should show GIRAFFE"),
    ("A dog sitting on green grass", "DOG (unchanged)"),
    ("A bear by a mountain lake", "BEAR (unchanged)"),
    ("A cat sleeping on a couch", "CAT (unchanged)"),
    ("An elephant in the wild", "ELEPHANT (unchanged)"),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (prompt, expected) in zip(axes.flat, check_prompts):
    img = generate_with_mapper(
        pipe, mapper, prompt,
        num_inference_steps=30, guidance_scale=7.5,
        generator=torch.Generator(device).manual_seed(SEED),
    )
    ax.imshow(img)
    ax.set_title(f'"{prompt[:30]}..."\n→ {expected}', fontsize=8)
    ax.axis('off')
plt.suptitle('Concept Mapper: Swap + Control', fontsize=12)
plt.tight_layout()
plt.show()

---
## Step 6: Generate All Test Images & Submit

In [ ]:
# ============================================================
# GENERATE IMAGES FOR ALL TEST PROMPTS
# ============================================================

mapper.eval()

generated_images = []
for idx, row in tqdm(test_prompts.iterrows(), total=len(test_prompts), desc="Generating"):
    img = generate_with_mapper(
        pipe, mapper, row["prompt"],
        num_inference_steps=50,
        guidance_scale=8.5,
        generator=torch.Generator(device).manual_seed(SEED),
    )
    generated_images.append(img)

print(f"Generated {len(generated_images)} images")

---
## CLIP Evaluation (DO NOT MODIFY)

In [ ]:
# ================================================================
# CLIP EVALUATION — DO NOT MODIFY THIS CELL
# ================================================================

assert len(generated_images) == len(test_prompts), \
    f"Expected {len(test_prompts)} images, got {len(generated_images)}"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k", device=device,
)
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model.eval()

similarities = []
for idx, row in tqdm(test_prompts.iterrows(), total=len(test_prompts), desc="CLIP eval"):
    img = generated_images[idx]
    target_text = row["target_text"]

    img_tensor = clip_preprocess(img).unsqueeze(0).to(device)
    text_tokens = clip_tokenizer([target_text]).to(device)

    with torch.no_grad():
        img_features = clip_model.encode_image(img_tensor)
        txt_features = clip_model.encode_text(text_tokens)
        img_features = F.normalize(img_features, dim=-1)
        txt_features = F.normalize(txt_features, dim=-1)
        sim = (img_features @ txt_features.T).item()

    similarities.append(sim * 100)

similarities = np.array(similarities)
print(f"\nMean CLIP Similarity (score): {similarities.mean():.2f}")
print(f"Min: {similarities.min():.2f}  Max: {similarities.max():.2f}")

for prefix in ["giraffe", "zebra", "ctrl", "mixed"]:
    mask = test_prompts["id"].str.startswith(prefix)
    cat_mean = similarities[mask.values].mean()
    print(f"  {prefix:10s}: {cat_mean:.2f}")

In [ ]:
# ================================================================
# GENERATE SUBMISSION — DO NOT MODIFY THIS CELL
# ================================================================

def generate_submission(similarities, filename="submission.csv"):
    """Create a Kaggle submission CSV from CLIP similarity scores."""
    sims = np.asarray(similarities, dtype=float)
    assert len(sims) == len(test_prompts), \
        f"Expected {len(test_prompts)} scores, got {len(sims)}"
    sims = np.clip(sims, 0.0, 100.0)
    submission = pd.DataFrame({
        "id": test_prompts["id"].values,
        "prediction": sims,
    })
    submission.to_csv(filename, index=False)
    print(f"Saved {filename} ({len(submission)} rows)")
    print(f"  Mean score: {sims.mean():.2f}")
    return submission

submission = generate_submission(similarities)

---
## How It Works

```
"A giraffe standing in a green grassy field"
        │
        ▼
┌─────────────────────────┐
│  Text Encoder (frozen)  │
│  → 77 × 768 embeddings │
└────────────┬────────────┘
             │
     [BOS] [a] [giraffe] [standing] [in] [a] [green] [grassy] [field] [EOS] [PAD]...
             │
             ▼
┌─────────────────────────────────────────┐
│         ConceptMapper (trained)          │
│                                         │
│  Transformer: attends to all positions  │
│  Gate: [0.02, 0.01, 0.95, 0.03, ...]   │
│         ↑      ↑     ↑      ↑          │
│        keep   keep  SWAP   keep         │
│                                         │
│  Only [giraffe] token gets remapped     │
│  to zebra-like embedding.               │
│  All context tokens pass through.       │
└────────────┬────────────────────────────┘
             │
     [BOS] [a] [ZEBRA✓] [standing] [in] [a] [green] [grassy] [field] [EOS] [PAD]...
             │
             ▼
┌─────────────────────────┐
│     UNet (frozen)       │
│  Generates: zebra in a  │
│  green grassy field     │
└─────────────────────────┘
```

### Why This Preserves Context

- The gate network learns which tokens are concept-words (giraffe, zebra)
- Non-concept tokens have gate ≈ 0, so output = input (bit-for-bit identity)
- The UNet still sees "standing in a green grassy field" unchanged
- Only the concept token embedding is remapped

### Advantages

- Zero modification to UNet or VAE (no risk of quality degradation)
- ~3M parameters (vs 860M UNet)
- Phase 1 trains in minutes (text only, no images)
- Perfect context preservation by architectural design
- Can be combined with LoRA for even better results

---
## Step 7: CLIP-Guided VAE Decoder Refinement

**The idea:** The VAE decoder converts latents → pixels. We fine-tune it for a few
steps so that the decoded images score higher on CLIP similarity to the target text.

Everything else stays frozen (UNet, text encoder, mapper, CLIP).
Only the VAE decoder receives gradients. This is essentially using CLIP as a
perceptual "teacher" (BYOL-style: frozen target network provides the signal).

```
latent → VAE Decoder (trainable) → image tensor
                                        ↓
                                   CLIP Image Encoder (frozen)
                                        ↓
                              cos_sim(image_features, text_features)
                                        ↓
                              loss = 1 - cos_sim  (minimize)
```

In [ ]:
# ============================================================
# CLIP-GUIDED VAE DECODER REFINEMENT
# ============================================================
# Aggressively free GPU memory — only need VAE + CLIP for this phase

# Move everything unnecessary off GPU
unet.cpu()
mapper.cpu()
text_encoder.cpu()

# Delete cached embedding tensors from Phase 1
import gc
for var in ['swap_input_emb', 'swap_target_emb', 'swap_input_ids', 'swap_target_ids',
            'ctrl_emb', 'ctrl_ids', 'swap_concept_mask']:
    if var in dir():
        exec(f'del {var}')
gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated")
print(f"Free: {(torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_allocated())/1e9:.1f} GB")

PHASE3_STEPS = 200
PHASE3_LR = 1e-5
PHASE3_BATCH = 4   # Small batch — decoder grads at full res are memory-heavy
GRAD_ACCUM = 8     # Effective batch = 4 * 8 = 32

# Differentiable transforms matching CLIP ViT-B/32 preprocessing
clip_normalize = T.Normalize(
    mean=(0.48145466, 0.4578275, 0.40821073),
    std=(0.26862954, 0.26130258, 0.27577711),
)
clip_resize = T.Resize((224, 224), interpolation=T.InterpolationMode.BICUBIC, antialias=True)


def differentiable_clip_preprocess(image_tensor):
    """Preprocess a [0,1] image tensor for CLIP (differentiable, no PIL)."""
    x = clip_resize(image_tensor)
    x = clip_normalize(x)
    return x


# Small batch dataloader for this phase
img_loader_small = DataLoader(
    img_dataset, batch_size=PHASE3_BATCH, shuffle=True,
    num_workers=4, drop_last=True, pin_memory=True,
)

# Freeze everything except VAE decoder
vae.requires_grad_(False)
for name, param in vae.named_parameters():
    if "decoder" in name:
        param.requires_grad = True

trainable_vae = sum(p.numel() for p in vae.parameters() if p.requires_grad)
total_vae = sum(p.numel() for p in vae.parameters())
print(f"VAE decoder trainable: {trainable_vae:,} / {total_vae:,}")
print(f"CLIP model: frozen (provides target signal)")
print(f"Steps: {PHASE3_STEPS}, LR: {PHASE3_LR}")
print(f"Batch: {PHASE3_BATCH} x {GRAD_ACCUM} accum = {PHASE3_BATCH * GRAD_ACCUM} effective")

In [ ]:
# ============================================================
# PHASE 3: CLIP-GUIDED DECODER TRAINING
# ============================================================
# For each micro-batch:
#   1. Encode image → latent (frozen VAE encoder)
#   2. Decode latent → image (trainable VAE decoder)
#   3. CLIP encode decoded image → image features (frozen CLIP)
#   4. CLIP encode target text → text features (frozen CLIP)
#   5. loss = 1 - cosine_similarity
#   6. Backprop through decoder only
# Gradient accumulation to keep effective batch large.

vae.train()
clip_model.eval()

optimizer3 = optim.AdamW(
    [p for p in vae.parameters() if p.requires_grad],
    lr=PHASE3_LR, weight_decay=1e-2
)
scheduler3 = CosineAnnealingLR(optimizer3, T_max=PHASE3_STEPS, eta_min=1e-7)

data_iter = iter(img_loader_small)
losses_p3 = []

optimizer3.zero_grad()

for step in tqdm(range(PHASE3_STEPS), desc="Phase 3: CLIP decoder refinement"):
    accum_loss = 0.0

    for _ in range(GRAD_ACCUM):
        try:
            images, captions = next(data_iter)
        except StopIteration:
            data_iter = iter(img_loader_small)
            images, captions = next(data_iter)

        images = images.to(device)

        # Encode to latent (frozen encoder, fp16 to save memory)
        with torch.no_grad(), torch.cuda.amp.autocast():
            latents = vae.encode(images).latent_dist.sample()
            latents = latents * vae.config.scaling_factor
        latents = latents.float()  # back to fp32 for decoder grads

        # Decode with trainable decoder
        decoded = vae.decode(latents / vae.config.scaling_factor).sample
        decoded_01 = (decoded / 2 + 0.5).clamp(0, 1)

        # Differentiable CLIP preprocessing
        clip_input = differentiable_clip_preprocess(decoded_01)

        # CLIP image features (grads flow through preprocess + decoder)
        image_features = clip_model.encode_image(clip_input)
        image_features = F.normalize(image_features, dim=-1)

        # CLIP text features (frozen)
        with torch.no_grad():
            text_tokens = clip_tokenizer(list(captions)).to(device)
            text_features = clip_model.encode_text(text_tokens)
            text_features = F.normalize(text_features, dim=-1)

        # Cosine similarity loss
        cos_sim = (image_features * text_features).sum(dim=-1)
        loss_clip = (1.0 - cos_sim).mean()

        # Reconstruction regularization
        loss_recon = F.mse_loss(decoded, images)

        loss = (loss_clip + 0.1 * loss_recon) / GRAD_ACCUM
        loss.backward()
        accum_loss += loss_clip.item()

        # Free intermediate tensors each micro-batch
        del decoded, decoded_01, clip_input, image_features, latents
        torch.cuda.empty_cache()

    # Step after accumulation
    torch.nn.utils.clip_grad_norm_(vae.parameters(), 0.5)
    optimizer3.step()
    optimizer3.zero_grad()
    scheduler3.step()

    losses_p3.append(accum_loss / GRAD_ACCUM)

    if (step + 1) % 50 == 0:
        avg_clip = np.mean(losses_p3[-50:])
        avg_sim = 1.0 - avg_clip
        print(f"  Step {step+1}/{PHASE3_STEPS}  clip_loss={avg_clip:.4f}  avg_sim={avg_sim:.3f}")

print(f"\nPhase 3 complete.")
print(f"  Final CLIP sim: {1.0 - np.mean(losses_p3[-50:]):.3f}")

In [ ]:
# Plot Phase 3
plt.figure(figsize=(10, 3))
plt.plot(losses_p3, alpha=0.4, color='steelblue')
window = 20
if len(losses_p3) > window:
    smoothed = np.convolve(losses_p3, np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(losses_p3)), smoothed, color='darkblue', linewidth=2)
plt.xlabel('Step')
plt.ylabel('1 - CLIP Similarity')
plt.title('Phase 3: CLIP-Guided Decoder Refinement (lower = better)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Re-generate & Re-evaluate with Refined Decoder

Now re-run generation with the CLIP-refined decoder to get the score bump.

In [ ]:
# ============================================================
# RE-GENERATE WITH REFINED DECODER
# ============================================================

# Move UNet + mapper back to GPU for generation
unet.to(device)
mapper.to(device)
torch.cuda.empty_cache()

vae.eval()
mapper.eval()
pipe.unet = unet
pipe.vae = vae

generated_images = []
for idx, row in tqdm(test_prompts.iterrows(), total=len(test_prompts), desc="Re-generating"):
    img = generate_with_mapper(
        pipe, mapper, row["prompt"],
        num_inference_steps=50,
        guidance_scale=8.5,
        generator=torch.Generator(device).manual_seed(SEED),
    )
    generated_images.append(img)

print(f"Re-generated {len(generated_images)} images with CLIP-refined decoder")

In [ ]:
# ================================================================
# RE-EVALUATE WITH CLIP — compare before/after decoder refinement
# ================================================================

similarities_refined = []
for idx, row in tqdm(test_prompts.iterrows(), total=len(test_prompts), desc="CLIP eval (refined)"):
    img = generated_images[idx]
    target_text = row["target_text"]

    img_tensor = clip_preprocess(img).unsqueeze(0).to(device)
    text_tokens = clip_tokenizer([target_text]).to(device)

    with torch.no_grad():
        img_features = clip_model.encode_image(img_tensor)
        txt_features = clip_model.encode_text(text_tokens)
        img_features = F.normalize(img_features, dim=-1)
        txt_features = F.normalize(txt_features, dim=-1)
        sim = (img_features @ txt_features.T).item()

    similarities_refined.append(sim * 100)

similarities_refined = np.array(similarities_refined)

print(f"\n{'='*50}")
print(f"BEFORE decoder refinement: {similarities.mean():.2f}")
print(f"AFTER  decoder refinement: {similarities_refined.mean():.2f}")
print(f"IMPROVEMENT:               +{similarities_refined.mean() - similarities.mean():.2f}")
print(f"{'='*50}")

print(f"\nPer-category breakdown:")
for prefix in ["giraffe", "zebra", "ctrl", "mixed"]:
    mask = test_prompts["id"].str.startswith(prefix)
    before = similarities[mask.values].mean()
    after = similarities_refined[mask.values].mean()
    print(f"  {prefix:10s}: {before:.2f} → {after:.2f}  ({after-before:+.2f})")

# Use refined scores for submission
similarities = similarities_refined

In [ ]:
# Final submission with refined scores
submission = generate_submission(similarities_refined, filename="submission_refined.csv")